In [ ]:
import os
from pathlib import Path
root = str(Path.cwd().parent.parent)
import sys
sys.path.append(f'{root}/src')

import pandas as pd
import constants as cs

import numpy as np
from scipy.interpolate import RegularGridInterpolator
import constants as cs
import netCDF4 as nc
import xarray as xr
from pygmt.params import Box, Position
import pyproj
import geopandas as gpd
import pygmt

In [ ]:
ice = xr.open_dataset(f'{root}/inputs/iceHistories/iceHistory-CISLGM_icepc_022226_highres_512.nc')

In [ ]:
times = [34, 30, 25, 20, 18, 17]

In [ ]:
### read in flowlines
jdf = pd.read_excel(f'{root}/inputs/flowlines/jdf_update.xlsx')
sv = pd.read_excel(f'{root}/inputs/flowlines/sv.xlsx')
ysv = pd.read_excel(f'{root}/inputs/flowlines/ysv_update.xlsx')
flowlines = [jdf, sv, ysv]

In [ ]:
clrs = [cs.clr_s, cs.clr_c, cs.clr_n]

In [ ]:
fig = pygmt.Figure()
fs = 12

w = 15
h = 11
nrow = 2
ncol = 3

region = [210,240, 45, 65]
projection = f'M0/0/{w/ncol*.75}c'

### loading in topo data
topo = pygmt.datasets.load_earth_relief(
    resolution="05m", 
    region=region, 
)

x_text = region[0] #+ 0.02 * (xmax - xmin)
y_text = region[-1] #- 0.02 * (ymax - ymin)

with fig.subplot(nrows=nrow, ncols=ncol, figsize=(f"{w}c", f"{h}c"), frame="lrtb"):
    for i in np.arange(nrow):
        for j in np.arange(ncol):
            ind = i*ncol + j
            with fig.set_panel(panel=ind):

                ### plotting topo
                pygmt.makecpt(cmap="gmt/globe+h", series=[-8000, 2000])
                fig.grdimage(grid=topo, region=region, projection=projection)
                if ind == 4:
                    with pygmt.config(FONT_ANNOT_PRIMARY=f"{fs}p", FONT_ANNOT_SECONDARY=f"{fs}p"):
                        fig.colorbar(frame = ['a2000f500', "x+lElevation (m)"], position=Position("BC", 'outside', offset = (-.5,-.7), anchor = 'MC'), 
                        orientation = 'horizontal', length = '100%')
                
                ### plotting coastline
                fig.coast(shorelines='1/0.5p,black', resolution = 'low',region=region, projection=projection, frame = ['af', 'WSne'])
                
                ### plotting ice
                pygmt.makecpt(cmap="SCM/devon", series=[0,4000, 100])
                fig.grdimage(grid=ice['ice_thickness'].sel(time = times[ind]), region=region, projection=projection, nan_transparent=True)
                if ind == 5:
                    with pygmt.config(FONT_ANNOT_PRIMARY=f"{fs}p", FONT_ANNOT_SECONDARY=f"{fs}p"):
                        fig.colorbar(frame = ['a1000f500', "x+lIce Thickness (m)"], position=Position("TR", 'outside', offset = (0,0), anchor = 'MC'), 
                        orientation = 'vertical', length = '100%')

                ### plotting text
                fig.text(x = x_text, y = y_text, text=f"{int(times[ind])} ka", fill="white", justify = 'TL', pen='1p,black', region = region, projection = projection)

                for n, flow in enumerate(flowlines):
                    fig.plot(x = flow['Lon'], y = flow['Lat'], pen = f'{4}p,white', projection = projection, region=region)
                    fig.plot(x = flow['Lon'], y = flow['Lat'], pen = f'{2}p,{clrs[n]}', projection = projection, region=region)
                    
                
                

fig.show()

# fig.savefig(f'{root}/figures/sf_iceGrowth.pdf')